# Praxisv04 Colab Experiment Runner

This notebook turns the original `APT_Praxis_Colab_Experiment.py` into a modular Praxisv04 workflow.

## Purpose Of This Notebook

- run the original experiment block-by-block inside Colab
- keep each block modular so individual model sections can be rerun without redoing every model
- show the purpose of each block before execution
- preserve the experiment outputs, visuals, and interpretation notes from the source experiment

## How To Use It

1. Open this notebook from the repo root in Colab.
2. Run the setup cell once.
3. See `PRAXISV04_COLAB_RUNBOOK.md` for the recommended baseline-only, novelty-only, and full-paper execution orders.
4. Run blocks `0-8` to prepare the environment, data, splits, graphs, and tracker.
5. Rerun any model block `9-18` independently as needed.
6. Run block `19` whenever you want refreshed comparison tables and visuals.

This notebook is self-contained for Colab upload: the setup cell writes the Praxisv04 runner and the source experiment into the runtime automatically.


In [ ]:
import base64
import sys
from pathlib import Path

RUNTIME_ROOT = Path("/content/praxisv04_runtime")
SRC_ROOT = RUNTIME_ROOT / "src"
PKG_ROOT = SRC_ROOT / "praxis"
REF_ROOT = RUNTIME_ROOT / "references"
PKG_ROOT.mkdir(parents=True, exist_ok=True)
REF_ROOT.mkdir(parents=True, exist_ok=True)

(PKG_ROOT / "__init__.py").write_text("", encoding="utf-8")
(PKG_ROOT / "praxisv04.py").write_text(
    base64.b64decode('...').decode("utf-8"),
    encoding="utf-8",
)
(REF_ROOT / "APT_Praxis_Colab_Experiment.py").write_text(
    base64.b64decode('...').decode("utf-8"),
    encoding="utf-8",
)

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from praxis.praxisv04 import load_default_runner

runner = load_default_runner(RUNTIME_ROOT)
runner.display_catalog()


## Modular Rerun Guide

- Blocks `0-8` are the shared preparation pipeline.
- Blocks `9-18` are model or decision blocks and can be rerun independently after preparation is complete.
- Block `15` prepares the sequence datasets used by blocks `17` and `18`.
- Block `19` rebuilds the final comparison visuals and tables from the current tracker state and is intended for full-paper runs.


## Block 00: INSTALLATION

**Purpose Of The Code**  
Bootstraps the Colab environment so every later block has the libraries it expects.

**Purpose**  
Install all required libraries. Run once per Colab session. mambapy is pure-PyTorch Mamba — avoids CUDA kernel issues on Colab. torch-geometric provides GNN layers and the ImbalancedSampler.

**Expected Outputs**  
Package install log, PyTorch version, and CUDA availability.

**How To Read The Output**  
Confirms the Colab runtime has graph, sequence, tuning, and explainability dependencies before any experiment logic runs.

**Recommended Prerequisites**  `None`

**Modular Rerun Note**  
Safe to rerun when the Colab runtime restarts or a package install fails.


In [ ]:
runner.run_block(0)


## Block 01: CONFIGURATION, SEEDS & GOOGLE DRIVE MOUNT

**Purpose Of The Code**  
Centralizes reproducibility, storage, and experiment-wide configuration in one place.

**Purpose**  
Set every random seed identically across all libraries so results are fully reproducible. Mount Drive for Optuna study persistence across Colab sessions (studies survive runtime disconnects).

**Expected Outputs**  
Seed confirmation, device selection, Google Drive mount, storage paths, and experiment constants.

**How To Read The Output**  
Shows whether the run is using GPU, where persistent artifacts will be saved, and which global hyperparameters govern the experiment.

**Recommended Prerequisites**  `0`

**Modular Rerun Note**  
Safe to rerun when you want to change paths or Colab storage settings.


In [ ]:
runner.run_block(1)


## Block 02: DATA LOADING

**Purpose Of The Code**  
Loads the full Unraveled dataset into a single analysis-ready frame while preserving temporal provenance.

**Purpose**  
Load all Unraveled CSV files from the network-flows directory, merge them into a single DataFrame, and perform initial validation. The dataset is organised as Week{N}/Day{M}/*.csv files. Upload your Unraveled data to Google Drive at the DATA_ROOT path above, keeping the original Week/Day folder structure intact.

**Expected Outputs**  
df_raw — full merged DataFrame with raw features and APT_Stage label.

**How To Read The Output**  
Confirms the dataset was read successfully and that the kill-chain labels align with the five target stages.

**Recommended Prerequisites**  `1`

**Modular Rerun Note**  
Rerun this block only when the dataset path or source files change.


In [ ]:
runner.run_block(2)


## Block 03: EXPLORATORY DATA ANALYSIS (EDA)

**Purpose Of The Code**  
Explains the raw data before modeling through class balance, temporal behavior, and feature-separation visuals.

**Purpose**  
Understand the dataset before any modelling. Visualise class imbalance, feature distributions, temporal patterns, and correlations. These plots motivate every modelling decision.

**Expected Outputs**  
8 plots covering class distribution, temporal flow patterns, feature correlations, and pairwise stage separation.

**How To Read The Output**  
Included under each plot.

**Recommended Prerequisites**  `2`

**Modular Rerun Note**  
Safe to rerun for fresh visuals without retraining any models.


In [ ]:
runner.run_block(3)


## Block 04: FEATURE TREATMENT

**Purpose Of The Code**  
Transforms the raw flow table into a consistent numerical feature space for downstream tabular, graph, and sequence models.

**Purpose**  
Remove identity/leakage features, encode time cyclically, and apply RobustScaler. This block is the most critical preprocessing step — the 0.9814 Mamba F1 before strict splits was caused by IP/timestamp leakage. Every item in DROP_COLS encodes WHO, not HOW.

**Expected Outputs**  
df_clean  — cleaned DataFrame with safe features only feature_cols — list of final feature column names scaler    — fitted RobustScaler (fitted on TRAIN only in Block 5)

**How To Read The Output**  
Shows how raw network-flow data is converted into a model-ready table while reducing leakage and preserving temporal signal.

**Recommended Prerequisites**  `2`

**Modular Rerun Note**  
Rerun this block when you change preprocessing logic or feature inclusion rules.


In [ ]:
runner.run_block(4)


## Block 05: TEMPORAL BLOCK SPLIT WITH STAGE STRATIFICATION

**Purpose Of The Code**  
Builds the temporally valid experimental split that the rest of the benchmark depends on.

**Purpose**  
Create train/val/test splits that respect APT temporal causality. A random 75/15/15 split would put DE effects in test and their causal predecessors in train — producing artificially high F1. Strict temporal separation ensures test only sees future campaigns.

**Expected Outputs**  
df_train, df_val, df_test + split summary table + visual

**How To Read The Output**  
Verifies that temporal causality is respected and that rare APT stages remain represented in all evaluation splits.

**Recommended Prerequisites**  `4`

**Modular Rerun Note**  
Rerun when you adjust split policy or want to inspect class coverage again.


In [ ]:
runner.run_block(5)


## Block 06: GRAPH CONSTRUCTION

**Purpose Of The Code**  
Constructs graph-structured training data from sequential flow windows.

**Purpose**  
Convert tabular flow windows into PyG Data objects with KNN edges. Each graph covers FLOWS_PER_WIN (512) consecutive flows within a capture_day window. Node features are the scaled flow statistics. Edges connect the K nearest flows in feature space. flows — e.g., a Foothold flow that PRECEDES a Lateral Movement flow will have similar byte/port signatures and be connected in the KNN graph. This relational context is what MLP cannot exploit.

**Expected Outputs**  
train_graphs, val_graphs, test_graphs — lists of PyG Data objects

**How To Read The Output**  
Confirms the tabular split was successfully converted into graph windows for the graph models.

**Recommended Prerequisites**  `5`

**Modular Rerun Note**  
Rerun when you change graph window size, stride, or KNN graph settings.


In [ ]:
runner.run_block(6)


## Block 07: LOSS FUNCTIONS

**Purpose Of The Code**  
Defines the shared objective functions used to train and compare the models.

**Purpose**  
Define the three loss components used in this praxis. These are the NOVEL contributions that no APT paper has applied. are not drowned. At γ=2, a 90%-confident prediction has 100× reduced loss contribution. more than over-estimating stage. No published APT paper uses this. 3. Monotonic Penalty (GMR-1): Penalises predicting DE without prior Foothold evidence in the recent window. Encodes kill-chain causality directly into the training signal.

**Expected Outputs**  
Class-balanced focal loss, kill-chain distance loss, monotonic penalty, and training class-count summary.

**How To Read The Output**  
Shows how imbalance handling and kill-chain-aware supervision are encoded before model training begins.

**Recommended Prerequisites**  `6`

**Modular Rerun Note**  
Rerun when you change beta, gamma, or kill-chain penalty settings.


In [ ]:
runner.run_block(7)


## Block 08: RESULTS TRACKER & COMPARISON TABLE

**Purpose Of The Code**  
Initializes shared experiment tracking so each model block can be trained and evaluated independently.

**Purpose**  
Central results store that every model block writes to. After each model runs, its per-stage F1 scores are added here. The final comparison table and visualisation are auto-generated. results_tracker.add(model_name, stage_f1_dict, macro_f1, pr_auc)

**Expected Outputs**  
Central results tracker object, comparison table scaffold, and model-comparison visualization hooks.

**How To Read The Output**  
Creates the single source of truth that every model block writes to, which makes reruns modular instead of forcing the whole benchmark to repeat.

**Recommended Prerequisites**  `7`

**Modular Rerun Note**  
Run this once before model blocks. After that, individual model blocks can be rerun independently.


In [ ]:
runner.run_block(8)


## Block 09: MLP BASELINE + SHAP ANALYSIS

**Purpose Of The Code**  
Establishes the tabular baseline and interpretable feature-importance benchmark.

**Purpose**  
The MLP is the non-graph tabular baseline. It evaluates each flow independently (no graph or temporal context). In Praxisv03 it achieved the best overall result (Macro F1=0.9535, DE F1=0.9575), proving the feature set IS discriminative for all stages. MLP success = feature signal exists. Graph model failure = graph chunking dilutes that signal. This finding motivates kill-chain-aware graph architectures. SHAP analysis identifies which features drive each stage prediction, directly informing which features APT-MAMBA and KC-CWT should preserve.

**Expected Outputs**  
MLP metrics, confusion matrix, SHAP visuals, and tracker updates.

**How To Read The Output**  
Provides the non-graph baseline and feature-attribution reference against which all graph and sequence models are judged.

**Recommended Prerequisites**  `8`

**Modular Rerun Note**  
Can be rerun independently after blocks 0-8 are complete.


In [ ]:
runner.run_block(9)


## Block 10: GATv2 (Graph Attention Network v2)

**Purpose Of The Code**  
Benchmarks an attention-based graph classifier on the shared graph windows.

**Purpose**  
GATv2 fixes the static attention bug of GATv1 — attention scores now depend on BOTH source and target node features simultaneously (dynamic attention), not just their concatenation at initialisation. This matters for APT because the relevance of a Foothold flow to a DE flow changes depending on the DE flow's current features. In this praxis GATv2 serves two roles: 1. Node-level classifier in the 5-class single-stage experiment 2. Stage 1 binary detector (if Decision Gate triggers two-stage) flows most influenced each stage prediction — a key XAI artifact.

**Expected Outputs**  
GATv2 training curves, confusion matrix, Optuna result, and tracker updates.

**How To Read The Output**  
Shows how attention-based graph modeling performs on the same split and where it succeeds or fails by stage.

**Recommended Prerequisites**  `8`

**Modular Rerun Note**  
Can be rerun independently after blocks 0-8 are complete.


In [ ]:
runner.run_block(10)


## Block 11: R-GCN (Relational Graph Convolutional Network)

**Purpose Of The Code**  
Tests whether explicit edge semantics improve graph reasoning over APT traffic.

**Purpose**  
R-GCN introduces typed edge relationships — different edge types use different weight matrices. For APT detection this means flows connected by "same destination port" use different aggregation than flows connected by "same protocol". This is important because the kill-chain pattern (Recon scanning port 22 → Foothold on port 22 → LM) produces a TYPED relational signature. R-GCN also served as the best DAPT-2020 model (F1=94.7%) in the original praxis, making it a critical baseline comparison.

**Expected Outputs**  
R-GCN training curves, confusion matrix, Optuna result, and tracker updates.

**How To Read The Output**  
Shows whether typed graph relations help recover stage-specific signal, especially for later kill-chain behavior.

**Recommended Prerequisites**  `8`

**Modular Rerun Note**  
Can be rerun independently after blocks 0-8 are complete.


In [ ]:
runner.run_block(11)


## Block 12: GIN (Graph Isomorphism Network)

**Purpose Of The Code**  
Benchmarks a structure-sensitive graph model that emphasizes subgraph discrimination.

**Purpose**  
GIN achieves maximum Weisfeiler-Leman expressiveness — it can distinguish the widest variety of graph substructures of any GNN. multisets of neighbours always produce distinct embeddings. h_v = MLP((1 + ε) · h_v + Σ h_u)  where ε is learnable GIN's PR-AUC of 0.5924 in Praxisv02 was the highest among GML models, suggesting the graph structure DOES contain discriminative signal — GIN just couldn't use temporal context to exploit it. This finding directly motivates APT-MAMBA and KC-CWT.

**Expected Outputs**  
GIN training curves, confusion matrix, Optuna result, and tracker updates.

**How To Read The Output**  
Shows how a high-expressivity structure-focused GNN behaves on the same graph windows.

**Recommended Prerequisites**  `8`

**Modular Rerun Note**  
Can be rerun independently after blocks 0-8 are complete.


In [ ]:
runner.run_block(12)


## Block 13: GCN-DGI (Deep Graph Infomax)

**Purpose Of The Code**  
Evaluates a self-supervised graph learning baseline before supervised fine-tuning.

**Purpose**  
DGI is self-supervised — it pretrains the encoder by maximising mutual information between LOCAL node representations and a GLOBAL graph summary. No labels required for pretraining. This is operationally important: in real APT detection, labeled attack flows are extremely scarce. DGI can leverage the abundant unlabeled Benign traffic during pretraining, then fine-tune on the small labeled APT subset. In Praxisv03, GCN-DGI was the BEST graph model at Macro F1=0.7456, achieving Recon F1=0.9554 and LM F1=0.9330. Only DE collapsed. PHASE 1 (pretraining): Unsupervised — learns node representations PHASE 2 (fine-tuning): Supervised — adds classification head

**Expected Outputs**  
DGI pretraining summary, downstream classifier metrics, confusion matrix, and tracker updates.

**How To Read The Output**  
Shows whether self-supervised graph pretraining improves downstream stage classification.

**Recommended Prerequisites**  `8`

**Modular Rerun Note**  
Can be rerun independently after blocks 0-8 are complete.


In [ ]:
runner.run_block(13)


## Block 14: ST-GCN (Spatial-Temporal GCN)

**Purpose Of The Code**  
Benchmarks a temporal graph baseline over the same flow windows.

**Purpose**  
ST-GCN is the ONLY temporal GML model in the baseline suite. It interleaves spatial graph convolution with 1D temporal convolution across window sequences. Its expected role: capture that Recon PRECEDES Foothold which PRECEDES LM — kill-chain temporal ordering. CRITICAL FINDING TO REPRODUCE: In every prior experiment, ST-GCN ranked LAST despite being the temporal model. This is the empirical proof that naive temporal convolution on 5-day data is insufficient. That failure is the DIRECT MOTIVATION for APT-MAMBA and KC-CWT. If ST-GCN ranks last here too, the doctoral argument is confirmed.

**Expected Outputs**  
ST-GCN metrics, training curves, confusion matrix, and tracker updates.

**How To Read The Output**  
Shows whether temporal graph modeling adds value beyond the static graph baselines.

**Recommended Prerequisites**  `8`

**Modular Rerun Note**  
Can be rerun independently after blocks 0-8 are complete.


In [ ]:
runner.run_block(14)


## Block 15: MAMBA BASELINE (No Kill-Chain Conditioning)

**Purpose Of The Code**  
Evaluates a pure sequence baseline over graph-derived flow sequences.

**Purpose**  
Standard Mamba (selective state space model) without the GMR-2 kill-chain modifications. This is the COMPARISON POINT for GMR-2. It proves that selective SSM alone improves on GML baselines, and the DELTA between Mamba Baseline and APT-MAMBA GMR-2 is the attribution for the kill-chain architectural novelty. Uses mambapy (pure PyTorch) — no CUDA kernel compilation needed. Processes kill-chain progression as a temporal sequence.

**Expected Outputs**  
Mamba metrics, training curves, confusion matrix, checkpoint, and tracker updates.

**How To Read The Output**  
Provides the sequence-model baseline without kill-chain conditioning and is the key comparison point for the later novel Mamba variant.

**Recommended Prerequisites**  `8`

**Modular Rerun Note**  
Can be rerun independently after blocks 0-8 are complete.


In [ ]:
runner.run_block(15)


## Block 16: DECISION GATE

**Purpose Of The Code**  
Turns the Phase 1 baseline results into a concrete go/no-go decision for the novel models.

**Purpose**  
Read DE F1 from all Phase 1 models and decide whether two-stage architecture is justified. This is the empirical checkpoint that prevents adding unnecessary complexity. The committee cannot question whether better tuning would have solved DE — this block provides the proof that properly tuned models were tried first.

**Expected Outputs**  
Printed decision + recommendation for next steps.

**How To Read The Output**  
Reads the current baseline results, especially Data Exfiltration behavior, and decides whether the evidence justifies moving on to the new models.

**Recommended Prerequisites**  `9, 10, 11, 12, 13, 14, 15`

**Modular Rerun Note**  
Rerun after any baseline model changes to refresh the novelty decision.


In [ ]:
runner.run_block(16)


## Block 17: APT-MAMBA GMR-2 (Kill-Chain Conditioned Mamba)

**Purpose Of The Code**  
Implements and evaluates the first novel contribution: kill-chain-conditioned Mamba.

**Purpose**  
The first primary novel contribution. WHAT CHANGES vs MAMBA BASELINE: Standard Mamba's B, C, Delta gates depend ONLY on current input x_t. GMR-2 conditions these gates on a kill-chain stage belief vector p_t derived from the previous hidden state: p_t = softmax(W_stage @ h_{t-1}.mean())   # stage belief B_t = Linear_B(concat[x_t, p_t])          # stage-conditioned φ_t = Σ w_k · p_t[k]  where w=[0,1,2,3,4] # amplifier Δ_t = softplus(Linear_D(concat[x_t, p_t]) + α·φ_t) increases → model retains more context for LM and DE windows. The kill-chain causality is baked into the gating mechanism. Also applies GMR-1 monotonic penalty in the combined loss.

**Expected Outputs**  
APT-Mamba metrics, training curves, confusion matrix, checkpoint, and tracker updates.

**How To Read The Output**  
Shows whether explicit kill-chain conditioning improves the Mamba baseline on later attack stages.

**Recommended Prerequisites**  `8, 15, 16`

**Modular Rerun Note**  
Can be rerun independently after blocks 0-8 and the baseline decision gate are complete.


In [ ]:
runner.run_block(17)


## Block 18: KC-CWT (Kill-Chain Causal Window Transformer)

**Purpose Of The Code**  
Implements and evaluates the second novel contribution: the kill-chain causal-window Transformer.

**Purpose**  
The second primary novel contribution — a Transformer alternative. THREE NOVEL COMPONENTS: (APT kill-chain is causal — DE happens after Recon, never before) 2. KILL-CHAIN STAGE BIAS: positive attention bias toward prior kill- chain stages. When classifying a DE window, Foothold windows receive amplified attention weight regardless of temporal distance. 3. MULTI-SCALE WINDOWS: small windows capture local burst patterns (Recon scanning), large windows capture global campaign progressions. Extended from DeepOP (2025) which validated causal window attention for MITRE ATT&CK sequences. The stage-bias component is UNPUBLISHED. KEY DIFFERENTIATOR vs APT-MAMBA: KC-CWT produces an attention heat map per prediction — showing WHICH prior windows drove the DE detection. This is the XAI artifact.

**Expected Outputs**  
KC-CWT metrics, attention heatmap, confusion matrix, checkpoint, and tracker updates.

**How To Read The Output**  
Shows whether a kill-chain-aware causal-window Transformer can outperform the baseline sequence and graph models.

**Recommended Prerequisites**  `8, 15, 16`

**Modular Rerun Note**  
Can be rerun independently after blocks 0-8 and the baseline decision gate are complete.


In [ ]:
runner.run_block(18)


## Block 19: FINAL COMPARISON TABLES & VISUALISATIONS

**Purpose Of The Code**  
Produces the full experiment summary so the benchmark can be reviewed, exported, and written up.

**Purpose**  
Generate the complete cross-model, cross-stage results that will appear in the praxis document. Every hypothesis is evaluated here. This block runs AFTER all model blocks are complete.

**Expected Outputs**  
Final comparison tables, per-stage heatmaps, DE-focused metrics, ablation waterfall, and saved CSV/PNG artifacts.

**How To Read The Output**  
Consolidates the complete benchmark into dissertation-ready comparison outputs across all baseline and novel models.

**Recommended Prerequisites**  `8, 9, 10, 11, 12, 13, 14, 15, 17, 18`

**Modular Rerun Note**  
Rerun after any model block to regenerate the final paper-ready tables and visuals.


In [ ]:
runner.run_block(19)


## Example Targeted Reruns

Use any of these in a new code cell after the notebook has been initialized:

```python
runner.run_block(15)  # rerun Mamba baseline only
runner.run_block(17)  # rerun APT-Mamba only
runner.run_block(18)  # rerun KC-CWT only
runner.run_block(19)  # regenerate final tables and visuals
```
